# COMP5329 — Deep Learning## Week 7 Self-Study: Sequence Modeling Architectures I — From RNN to Transformer**Semester 1, 2026**This self-study material provides a comprehensive walk-through of **all** topics covered in the Week 7 lecture. It is designed as a **standalone resource** — you do not need the tutorial or lecture slides alongside this document, though they remain useful references.**What this material covers:**1. Why sequential data needs specialised architectures2. Word representation methods for NLP3. Recurrent Neural Networks (RNN) — theory, implementation, and BPTT4. LSTM — gating, gradient analysis, stacked and bidirectional variants5. GRU — a parameter-efficient alternative6. RNN applications — VQA, Reading Comprehension, Seq2seq, Cross-Attention7. Transformer — self-attention, multi-head attention, positional encoding, full architecture8. Comprehensive comparison and exam-style questions

### Learning ObjectivesBy the end of this self-study you will be able to:1. Describe three word representation methods and their trade-offs.2. Derive the **BPTT** (Backpropagation Through Time) gradient for vanilla RNNs and explain vanishing/exploding gradients.3. Explain how LSTM's **additive cell-state update** solves the vanishing gradient problem, using the detailed $A_t + B_t + C_t + D_t$ gradient decomposition.4. Implement RNN, LSTM, and GRU from scratch and compare them on a shared benchmark.5. Describe Stacked LSTMs and Bidirectional RNNs and when to use them.6. Explain the **Seq2seq** encoder-decoder framework and how **cross-attention** improves it.7. Describe real-world RNN applications: VQA and Reading Comprehension.8. Derive the self-attention mechanism from first principles and explain the $\sqrt{d_k}$ scaling.9. Implement **multi-head attention**, **positional encoding**, and a **Transformer encoder block**.10. Understand the full Transformer encoder-decoder architecture, including masking.11. Visualise attention weights and interpret them.

---## 1. Introduction to Sequence Modeling### 1.1 Why Sequence Models?In previous weeks we studied CNNs (which assume **spatial locality** with fixed receptive fields) and GNNs (which assume **graph-structured** relationships). However, many real-world tasks — natural language, time series, music, DNA — involve data with **ordered, variable-length** dependencies. We need architectures that can process sequences while maintaining a summary of past context.| Architecture | Input processing | Memory mechanism | Parallelisable? ||---|---|---|---|| MLP | All at once (fixed size) | None | Yes || CNN | Local windows | Receptive field | Yes || **RNN** | One token at a time | Hidden state | No || **LSTM** | One token at a time | Gated cell state | No || **GRU** | One token at a time | Gated hidden state | No || **Transformer** | All at once | Attention weights | Yes |This material walks through the evolution of sequence modelling: from the simplest recurrent unit to the Transformer. Each successive architecture solves a specific limitation of its predecessor.

### 1.2 Word Representation MethodsBefore feeding text into a neural network, we need to convert words (or characters) into numerical vectors. The lecture introduces three approaches:#### One-Hot EncodingThe simplest representation. The vector length equals the vocabulary size. Each word is assigned a unique dimension set to 1; all other dimensions are 0.**Example** with vocabulary = {apple, bag, cat, dog, elephant}:| Word | $d_1$ | $d_2$ | $d_3$ | $d_4$ | $d_5$ ||---|---|---|---|---|---|| apple | 1 | 0 | 0 | 0 | 0 || bag | 0 | 1 | 0 | 0 | 0 || cat | 0 | 0 | 1 | 0 | 0 || dog | 0 | 0 | 0 | 1 | 0 || elephant | 0 | 0 | 0 | 0 | 1 |**Limitations:**- **High-dimensional**: A vocabulary of 100,000 words produces 100,000-dimensional vectors.- **No similarity**: All pairs of words are equidistant ($\|\mathbf{w}_i - \mathbf{w}_j\|_2 = \sqrt{2}$ for $i \neq j$). "cat" is as far from "dog" as it is from "economics".- **Sparse**: Each vector has only one non-zero entry.#### Word Hashing (Character n-gram Hashing)Instead of mapping entire words, we decompose each word into **character-level n-grams** (e.g., trigrams) and hash them into a fixed-size vector. For example, "apple" → {"a-p-p", "p-p-l", "p-l-e"}.**Advantages:**- Fixed-size representation regardless of vocabulary.- Can handle **out-of-vocabulary** words (since any new word can be decomposed into n-grams).- Dimensionality is much smaller: with $26^3 = 17{,}576$ possible letter trigrams.**Limitation:** Collisions — different words may share the same n-gram profile.#### Word EmbeddingLearned dense vectors of moderate dimensionality (e.g., 50–300 dimensions). Trained so that semantically similar words have similar vectors.$$\text{embed}: \text{word} \to \mathbb{R}^d, \quad d \ll |V|$$**Key property**: Semantic relationships are encoded as vector arithmetic. The classic example:$$\text{king} - \text{man} + \text{woman} \approx \text{queen}$$In practice, an `nn.Embedding(vocab_size, embed_dim)` layer is a learnable lookup table that is trained end-to-end with the rest of the model.

In [ ]:
# ── Word Representation Demonstration ──────────────────────────────────────────import torchimport torch.nn as nnimport numpy as np# --- One-Hot Encoding ---vocab = ['apple', 'bag', 'cat', 'dog', 'elephant']word_to_idx = {w: i for i, w in enumerate(vocab)}def one_hot(word, vocab_size=len(vocab)):    vec = torch.zeros(vocab_size)    vec[word_to_idx[word]] = 1.0    return vecprint("One-hot encoding:")for w in vocab:    print(f"  {w:>10s} -> {one_hot(w).tolist()}")# Distances between one-hot vectorsprint(f"\n  dist(cat, dog)      = {torch.dist(one_hot('cat'), one_hot('dog')):.4f}")print(f"  dist(cat, elephant) = {torch.dist(one_hot('cat'), one_hot('elephant')):.4f}")print("  -> All pairs equidistant!")# --- Word Embedding ---print("\nWord embedding (randomly initialised):")embed = nn.Embedding(num_embeddings=len(vocab), embedding_dim=8)for w in vocab:    idx = torch.tensor([word_to_idx[w]])    vec = embed(idx).detach().squeeze()    print(f"  {w:>10s} -> [{', '.join(f'{v:.2f}' for v in vec[:4].tolist())}  ...]")print("  -> Dense, low-dimensional, trainable!")

---## 1.3 Data Pipeline — Letter-Counting TaskThroughout Sections 2–4, we use a shared benchmark to compare RNN, LSTM, and GRU. The task: given a random string of lowercase letters (a-z), uppercase letters (A-Z), and digit noise (0-9), predict the **difference** between the count of lowercase and uppercase letters (offset by 29 to keep labels non-negative).**Examples:**- `aAA304` → label = 1 - 2 + 29 = 28- `bbB234BbB` → label = 3 - 3 + 29 = 29- `ccccccC` → label = 6 - 1 + 29 = 34This requires the model to **count** across variable-length sequences while ignoring noise — a clean test of sequential memory.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────import osimport randomimport timeimport sysimport gcimport mathimport numpy as npimport matplotlib.pyplot as pltimport torchimport torch.nn as nnimport torch.optim as optimimport torch.nn.functional as Ffrom torch.autograd import Variable%matplotlib inline

In [ ]:
# ── Data generation ─────────────────────────────────────────────────────────────def generate_line(input_char):    """Generate a single training example."""    num1 = random.randint(1, 30)   # number of lowercase letters    num2 = random.randint(1, 30)   # number of uppercase letters    src = [chr(input_char) for _ in range(num1)]    target = [chr(input_char - 32) for _ in range(num2)]    src.extend(target)    noise_num = random.randint(0, 100)    for _ in range(noise_num):        src.append(str(random.randint(0, 9)))    random.shuffle(src)    return ''.join(src), num1 - num2 + 29def generate_data(size, filename):    """Generate dataset and write to file."""    f = open(filename, "w")    s = set()    count = 0    while count < size:        c = random.randint(ord('a'), ord('z'))        src, target = generate_line(c)        if src in s or src[::-1] in s:            continue        count += 1        if count % 10000 == 0:            print("Generated %d lines" % count)        s.add(src)        f.write('\t'.join([src, str(target)]))        f.write('\n')    f.close()generate_data(80000, "seq.txt")

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────────hidden_size = 50          # hidden state dimension for all RNN variantsembedding_size = 20       # character embedding dimensioninput_length = 160        # max sequence length (zero-padded)vob_size = 52 + 10 + 1    # 26 lower + 26 upper + 10 digits + 1 padding tokenoutput_size = 60          # number of classesmax_gradient_norm = 3     # gradient clipping thresholdinit_lr_rate = 0.001      # Adam learning rateMAX_ITERATIONS = 20000VAL_INTERVAL = 1000PRINT_INTERVAL = 100batch_size = 64

In [ ]:
# ── Data loading and preprocessing utilities ─────────────────────────────────def read_dataset(file_name):    f = open(file_name)    ls = []    for line in f.readlines():        line = line.strip()        l = line.split('\t')        ls.append([l[0], int(l[1])])    random.shuffle(ls)    return ls[:64000], ls[64000:]def create_maps():    dic = {}    counter = 1    for i in range(ord('a'), ord('z') + 1):        dic[chr(i)] = counter        counter += 1    for i in range(ord('A'), ord('Z') + 1):        dic[chr(i)] = counter        counter += 1    for i in range(ord('0'), ord('9') + 1):        dic[chr(i)] = counter        counter += 1    return dicdef word_embedding(input_seq, vob_size, dtype=torch.float):    word_embed = nn.Embedding(vob_size, embedding_size)    embeddings = word_embed(input_seq.long())    return embeddingsdef create_batch(datas, maps):    size = len(datas)    seqs = np.zeros((size, input_length), dtype=np.int32)    labels = np.zeros(size, dtype=np.int32)    for i in range(size):        labels[i] = datas[i][1]        seq = datas[i][0]        l = input_length - len(seq)        for j in range(len(seq)):            seqs[i][l + j] = maps[seq[j]]    return seqs, labelsmaps = create_maps()

In [ ]:
# ── Reusable training and evaluation pipeline ────────────────────────────────results = {}def train_and_log(model, model_name, maps):    train_data, val_data = read_dataset("seq.txt")    pointer = 0    model.train()    optimizer = optim.Adam(model.parameters(), lr=init_lr_rate)    model_loss = nn.CrossEntropyLoss()    train_losses, val_accs = [], []    running_loss = 0.0    for step in range(MAX_ITERATIONS + 1):        if pointer + batch_size >= len(train_data):            random.shuffle(train_data)            pointer = 0        datas = train_data[pointer:pointer + batch_size]        pointer += batch_size        input_seq, label = create_batch(datas, maps)        input_seq = torch.from_numpy(input_seq)        label = torch.from_numpy(label)        seq_emb = word_embedding(input_seq, vob_size)        input_seq_emb = seq_emb.permute(1, 0, 2)        out = model.predict(input_seq_emb)        step_loss = model_loss(out, label.long())        optimizer.zero_grad()        step_loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), max_gradient_norm)        optimizer.step()        running_loss += step_loss.item()        if step > 0 and step % PRINT_INTERVAL == 0:            avg_loss = running_loss / PRINT_INTERVAL            train_losses.append(avg_loss)            running_loss = 0.0            if step % (PRINT_INTERVAL * 10) == 0:                print(f"[{model_name}] step {step}, loss {avg_loss:.3f}")        if step % VAL_INTERVAL == 0:            model.eval()            val_rate = val_model(model, val_data, maps)            val_accs.append(val_rate)            model.train()            print(f"[{model_name}] step {step}, val accuracy: {val_rate:.3f}")    results[model_name] = {"train_loss": train_losses, "val_acc": val_accs}    print(f"\n[{model_name}] Training complete. Final val accuracy: {val_accs[-1]:.3f}")def val_model(model, dataset, maps):    start_pointer = 0    total = 0    with torch.no_grad():        while start_pointer < len(dataset):            end_pointer = min(start_pointer + batch_size, len(dataset))            datas = dataset[start_pointer:end_pointer]            start_pointer = end_pointer            input_seq, label = create_batch(datas, maps)            input_seq = torch.from_numpy(input_seq)            label = torch.from_numpy(label)            seq_emb = word_embedding(input_seq, vob_size)            input_seq_emb = seq_emb.permute(1, 0, 2)            answers = model.predict(input_seq_emb)            answer_ids = np.argmax(answers.detach().numpy(), axis=-1)            total += np.sum(label.detach().numpy() == answer_ids)    return 1.0 * total / len(dataset)

---## 2. Vanilla RNN### 2.1 TheoryThe simplest approach to sequence modelling: at each time step, combine the current input with a summary of everything seen so far.The core equations of a vanilla RNN are:$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h)$$$$y_t = W_{hy}\, h_t + b_y$$where:- $x_t \in \mathbb{R}^{d}$ is the input at time step $t$- $h_t \in \mathbb{R}^{H}$ is the hidden state (the "memory")- $W_{xh} \in \mathbb{R}^{H \times d}$, $W_{hh} \in \mathbb{R}^{H \times H}$ are learnable weight matrices- $h_0 = \mathbf{0}$ (initialised to zeros)**Key insight**: The hidden state $h_t$ is a **lossy compression** of the entire input history $x_1, \ldots, x_t$ into a fixed-size vector of $H$ dimensions. Everything the model knows about the past must fit in this vector.#### Numerical ExampleConsider a toy RNN with $H = 1$, $d = 1$, all weights = 1, no bias, and linear activation:| Time step | Input $x_t$ | $h_{t-1}$ | $h_t = h_{t-1} + x_t$ | Output $y_t = h_t$ ||---|---|---|---|---|| $t=1$ | 1 | 0 | 1 | 1 || $t=2$ | 2 | 1 | 3 | 3 || $t=3$ | 3 | 3 | 6 | 6 |With a second layer, the hidden states feed into another recurrence, enabling hierarchical feature extraction.In practice, we often concatenate $[x_t; h_{t-1}]$ and use a single weight matrix $W \in \mathbb{R}^{H \times (d+H)}$, which is mathematically equivalent to separate $W_{xh} x_t + W_{hh} h_{t-1}$.

### 2.2 From-Scratch Implementation

In [ ]:
# ── Vanilla RNN ────────────────────────────────────────────────────────────────class RNNModel(nn.Module):    def __init__(self, in_feature, hidden_size, n_class):        super(RNNModel, self).__init__()        self.in_feature = in_feature        self.hidden_size = hidden_size        self.n_class = n_class        self.fully_connected = nn.Linear(in_feature + self.hidden_size, self.hidden_size)        self.pred_layer = nn.Linear(self.hidden_size, self.n_class)        self.tanh = nn.Tanh()    def forward(self, input, dtype=torch.float):        T = input.shape[0]        batch_size = input.shape[1]        outputs = torch.zeros(size=(T, batch_size, self.hidden_size), dtype=dtype)        state = torch.zeros(size=(batch_size, self.hidden_size), dtype=dtype)        for t in range(T):            concat = torch.cat([input[t], state], dim=1)  # (B, d+H)            state = self.tanh(self.fully_connected(concat))            outputs[t] = state        return outputs, state    def predict(self, input_state, dtype=torch.float):        _, last_state = self.forward(input_state)        predict = self.tanh(self.pred_layer(last_state))        return predict

### 2.3 Training

In [ ]:
rnn_model = RNNModel(embedding_size, hidden_size, output_size)train_and_log(rnn_model, 'rnn', maps)

### 2.4 Backpropagation Through Time (BPTT)BPTT is the algorithm used to compute gradients in RNNs. After the RNN outputs predictions, we compute the error $E$ and backpropagate through the unrolled computation graph.For a sequence of $T$ time steps, the total gradient of the error with respect to the weights is:$$\frac{\partial E}{\partial W} = \sum_{t=1}^{T} \frac{\partial E_t}{\partial W}$$The weights are updated by gradient descent:$$W \leftarrow W - \alpha \frac{\partial E}{\partial W}$$#### Gradient DerivationThe gradient of the error at time step $k$ can be expanded using the chain rule:$$\frac{\partial E_k}{\partial W} = \frac{\partial E_k}{\partial h_k} \cdot \frac{\partial h_k}{\partial c_k} \cdot \frac{\partial c_k}{\partial c_{k-1}} \cdots \frac{\partial c_2}{\partial c_1} \cdot \frac{\partial c_1}{\partial W}$$where $c_t = W_{hh} h_{t-1} + W_{hx} x_t$ is the pre-activation. This can be written compactly as:$$\frac{\partial E_k}{\partial W} = \frac{\partial E_k}{\partial h_k} \cdot \frac{\partial h_k}{\partial c_k} \prod_{t=2}^{k} \frac{\partial c_t}{\partial c_{t-1}} \cdot \frac{\partial c_1}{\partial W}$$Now, since $c_t = W_{hh} \cdot \tanh(c_{t-1}) + W_{hx} x_t$, the derivative of $c_t$ with respect to $c_{t-1}$ is:$$\frac{\partial c_t}{\partial c_{t-1}} = \tanh'(c_{t-1}) \cdot W_{hh}$$Plugging this into the gradient expression:$$\frac{\partial E_k}{\partial W} = \frac{\partial E_k}{\partial h_k} \cdot \frac{\partial h_k}{\partial c_k} \left(\prod_{t=2}^{k} \tanh'(c_{t-1}) \cdot W_{hh}\right) \frac{\partial c_1}{\partial W}$$This product is the source of both **vanishing** and **exploding** gradients.

### 2.5 The Vanishing and Exploding Gradient ProblemThe critical term in the BPTT gradient is the product:$$\prod_{t=2}^{k} \tanh'(c_{t-1}) \cdot W_{hh}$$Since $|\tanh'(z)| \leq 1$ everywhere (and typically $\ll 1$), we have two cases:**Vanishing gradients** ($\|W_{hh}\| < 1$): The product **shrinks exponentially** with $k$:$$\prod_{t} \tanh'(\cdot) \cdot W_{hh} \to 0 \quad \text{as } k \to \infty$$Concrete example: $0.9^{1000} \approx 0$ — gradients from distant time steps vanish.**Exploding gradients** ($\|W_{hh}\|$ is large enough to overpower $\tanh'$): The product **grows exponentially**:$$\prod_{t} \tanh'(\cdot) \cdot W_{hh} \to \infty \quad \text{as } k \to \infty$$Concrete example: $1.01^{1000} \approx 21{,}000$ — gradients explode.**Consequences:**- **Vanishing**: The model **cannot learn long-range dependencies** — it effectively "forgets" distant inputs. Gradient clipping does **nothing** for vanishing gradients.- **Exploding**: Training becomes unstable with NaN losses. **Gradient clipping** (capping $\|\nabla\|$ at a threshold) is the standard fix.> **Transition**: We need a mechanism that allows gradients to flow across many time steps **without multiplicative decay**. The key idea: an **additive** update path. This leads us to LSTM.

---## 3. LSTM (Long Short-Term Memory)### 3.1 TheoryLSTM introduces a **cell state** $c_t$ that acts as a conveyor belt. Information flows along it with only **additive** modifications, not multiplicative. Three **gates** control what enters, what leaves, and what is forgotten. Think of the LSTM cell as having 4 inputs and 1 output:- **Input gate** — controls how much new information to write- **Forget gate** — controls how much old memory to keep- **Output gate** — controls how much memory to expose- The cell state is the internal memory| Gate | Formula | Role ||---|---|---|| **Forget** | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | What fraction of old cell state to **keep** || **Input** | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | What fraction of new candidate to **add** || **Candidate** | $\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$ | Proposed **new information** || **Cell update** | $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ | **Additive** update (the key!) || **Output** | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | What part of cell state to **expose** || **Hidden** | $h_t = o_t \odot \tanh(c_t)$ | Output hidden state |**Why these activation functions?**- **Sigmoid** ($\sigma$) for gates: outputs in $[0, 1]$ act as **soft switches** — 0 means "block everything", 1 means "pass everything through".- **Tanh** for candidate $\tilde{c}_t$: centered around 0, allowing the cell state to both increase and decrease.

### 3.2 Why LSTM Solves the Vanishing Gradient Problem — Detailed Analysis#### Simplified ArgumentThe cell state gradient across time is:$$\frac{\partial c_T}{\partial c_k} = \prod_{t=k+1}^{T} f_t$$If the forget gate learns $f_t \approx 1$, this gradient passes through **unchanged** — no exponential decay! The network **learns** to "not forget" when long-range memory is needed.#### Full Gradient Decomposition ($A_t, B_t, C_t, D_t$)The cell state update $c_t = c_{t-1} \odot f_t + \tilde{c}_t \odot i_t$ involves four terms that depend on $c_{t-1}$:$$\frac{\partial c_t}{\partial c_{t-1}} = \underbrace{\frac{\partial f_t}{\partial c_{t-1}} \cdot c_{t-1}}_{A_t} + \underbrace{f_t}_{B_t} + \underbrace{\frac{\partial \tilde{c}_t}{\partial c_{t-1}} \cdot i_t}_{C_t} + \underbrace{\frac{\partial i_t}{\partial c_{t-1}} \cdot \tilde{c}_t}_{D_t}$$The LSTM state gradient becomes:$$\frac{\partial E_k}{\partial W} = \frac{\partial E_k}{\partial h_k} \cdot \frac{\partial h_k}{\partial c_k} \left(\prod_{t=2}^{k} [A_t + B_t + C_t + D_t]\right) \frac{\partial c_1}{\partial W}$$**Why this helps:**1. **The forget gate term $B_t = f_t$**: The gradient contains the forget gate's activations directly. The network can learn to set $f_t \approx 1$, keeping gradients alive.2. **Additive structure**: The gradient is a **sum** of four terms, not a single multiplicative factor like in RNNs. Even if some terms vanish, the others can maintain gradient flow.3. **Adaptive control**: At each time step, the LSTM can learn parameter updates that keep the sum $A_t + B_t + C_t + D_t$ away from zero, preventing gradient collapse.**Contrast with RNN**: In RNNs, the gradient contains a **single multiplicative factor** $\tanh'(\cdot) \cdot W_{hh}$ at each step. All factors must stay close to 1 simultaneously — a much harder condition to satisfy.**Summary**: LSTMs solve vanishing gradients through:- **Additive** gradient structure (sum of four terms)- **Direct access** to forget gate activations in the gradient- **Learnable** gate values that can maintain gradient flow at each time step

### 3.3 From-Scratch Implementation

In [ ]:
# ── LSTM ──────────────────────────────────────────────────────────────────────class LSTMModel(nn.Module):    def __init__(self, in_feature, hidden_size, n_class):        super(LSTMModel, self).__init__()        self.in_feature = in_feature        self.hidden_size = hidden_size        self.n_class = n_class        self.fully_connected = nn.Linear(self.in_feature + self.hidden_size,                                          4 * self.hidden_size)        self.pred_layer = nn.Linear(self.hidden_size, self.n_class)        self.tanh = nn.Tanh()        self.sigmoid = nn.Sigmoid()    def forward(self, input, dtype=torch.float):        T = input.shape[0]        batch_size = input.shape[1]        outputs = torch.zeros(size=(T, batch_size, self.hidden_size), dtype=dtype)        c, h = torch.unbind(torch.zeros([2, batch_size, self.hidden_size]), dim=0)        for t in range(T):            concat = torch.cat([input[t], h], dim=1)    # (B, d+H)            gates = self.fully_connected(concat)         # (B, 4H)            i, f, g, o = gates.chunk(4, dim=1)           # each (B, H)            i_t = self.sigmoid(i)   # input gate            f_t = self.sigmoid(f)   # forget gate            g_t = self.tanh(g)      # candidate cell state            o_t = self.sigmoid(o)   # output gate            c = f_t * c + i_t * g_t  # cell state update (ADDITIVE!)            h = o_t * self.tanh(c)   # hidden state            outputs[t] = h        return outputs, h    def predict(self, input_state, dtype=torch.float):        _, last_state = self.forward(input_state)        predict = self.pred_layer(last_state)        return predict

### 3.4 Training and Comparison with RNN

In [ ]:
lstm_model = LSTMModel(embedding_size, hidden_size, output_size)train_and_log(lstm_model, 'lstm', maps)

In [ ]:
# ── RNN vs LSTM comparison ───────────────────────────────────────────────────fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))for name in ['rnn', 'lstm']:    if name in results:        ax1.plot(results[name]['train_loss'], label=name.upper())        ax2.plot(results[name]['val_acc'], label=name.upper())ax1.set_xlabel('Step (x{})'.format(PRINT_INTERVAL))ax1.set_ylabel('Training Loss')ax1.set_title('Training Loss: RNN vs LSTM')ax1.legend(); ax1.grid(True, alpha=0.3)ax2.set_xlabel('Validation checkpoint')ax2.set_ylabel('Accuracy')ax2.set_title('Validation Accuracy: RNN vs LSTM')ax2.legend(); ax2.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

### 3.5 LSTM vs RNN: Key Differences| Aspect | RNN | LSTM ||---|---|---|| **Memory** | Single hidden state $h_t$ | Separate cell state $c_t$ + hidden state $h_t$ || **Update rule** | $h_t = \tanh(W[h_{t-1}; x_t])$ (multiplicative) | $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ (additive) || **Parameters** | $H \times (H + d)$ | $4 \times H \times (H + d)$ — **4x more** || **Gradient flow** | $\prod_t \tanh'(\cdot) W_{hh}$ → vanishes | $\prod_t (A_t + B_t + C_t + D_t)$ → controlled || **Long-range** | Poor (gradient vanishes) | Good (forget gate ≈ 1 preserves gradients) |

### 3.6 Stacked LSTMA single LSTM layer processes the input sequence and produces a sequence of hidden states. A **stacked LSTM** feeds the hidden states from one layer as inputs to the next:$$h_t^{(l)} = \text{LSTM}^{(l)}(h_t^{(l-1)},\, h_{t-1}^{(l)})$$where $h_t^{(0)} = x_t$ (the input embedding) and $l$ indexes the layer.**Why stack?** Each layer extracts increasingly abstract features from the sequence, similar to how deeper CNN layers capture higher-level patterns. In practice, 2-4 layers are common; beyond that, returns diminish and training becomes harder.

In [ ]:
# ── Stacked LSTM demonstration ────────────────────────────────────────────────class StackedLSTM(nn.Module):    """Multi-layer LSTM using PyTorch's built-in nn.LSTM."""    def __init__(self, input_size, hidden_size, num_layers, n_class):        super().__init__()        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=False)        self.pred_layer = nn.Linear(hidden_size, n_class)    def forward(self, x):        # x: (T, B, d)        outputs, (h_n, c_n) = self.lstm(x)  # h_n: (num_layers, B, H)        return outputs, h_n[-1]  # use last layer's final hidden state    def predict(self, x):        _, last_h = self.forward(x)        return self.pred_layer(last_h)# Demonstrate shape flowstacked = StackedLSTM(embedding_size, hidden_size, num_layers=3, n_class=output_size)dummy_input = torch.randn(10, 4, embedding_size)  # T=10, B=4, d=20out, last_h = stacked(dummy_input)print(f"Input shape:           {dummy_input.shape}  (T, B, d)")print(f"All hidden states:     {out.shape}  (T, B, H)")print(f"Last layer final h:    {last_h.shape}  (B, H)")print(f"Parameters: {sum(p.numel() for p in stacked.parameters()):,}")

### 3.7 Bidirectional RNNA standard RNN only processes the sequence **left-to-right**, so $h_t$ only encodes information from $x_1, \ldots, x_t$. A **bidirectional RNN** adds a second pass **right-to-left**:$$\overrightarrow{h}_t = \text{RNN}_{\text{fwd}}(x_t, \overrightarrow{h}_{t-1})$$$$\overleftarrow{h}_t = \text{RNN}_{\text{bwd}}(x_t, \overleftarrow{h}_{t+1})$$$$h_t = [\overrightarrow{h}_t;\, \overleftarrow{h}_t]$$The final representation at each position concatenates both directions, giving a **2H-dimensional** vector that encodes the full sequence context.**When to use:**- NLP tasks where future context matters (e.g., named entity recognition, sentiment analysis)- **Not** suitable for autoregressive generation (can't use future tokens at inference time)- Doubles the parameter count of the recurrent layer

In [ ]:
# ── Bidirectional LSTM demonstration ──────────────────────────────────────────class BiLSTM(nn.Module):    def __init__(self, input_size, hidden_size, n_class):        super().__init__()        self.lstm = nn.LSTM(input_size, hidden_size, bidirectional=True, batch_first=False)        # Output dimension is 2*hidden_size due to bidirectional        self.pred_layer = nn.Linear(2 * hidden_size, n_class)    def forward(self, x):        outputs, (h_n, c_n) = self.lstm(x)        # h_n: (2, B, H) — forward and backward final states        # Concatenate forward and backward final hidden states        last_h = torch.cat([h_n[0], h_n[1]], dim=1)  # (B, 2H)        return outputs, last_h    def predict(self, x):        _, last_h = self.forward(x)        return self.pred_layer(last_h)bi_model = BiLSTM(embedding_size, hidden_size, output_size)dummy_input = torch.randn(10, 4, embedding_size)out, last_h = bi_model(dummy_input)print(f"Input shape:       {dummy_input.shape}  (T, B, d)")print(f"BiLSTM output:     {out.shape}  (T, B, 2H)")print(f"Concatenated h:    {last_h.shape}  (B, 2H)")print(f"Parameters: {sum(p.numel() for p in bi_model.parameters()):,}")

---## 4. GRU (Gated Recurrent Unit)### 4.1 TheoryGRU simplifies LSTM by merging the cell state and hidden state into a single $h_t$, and using **2 gates** instead of 3.| Gate | Formula | Role ||---|---|---|| **Reset** | $r_t = \sigma(W_r [h_{t-1}, x_t] + b_r)$ | How much past to forget when computing candidate || **Update** | $z_t = \sigma(W_z [h_{t-1}, x_t] + b_z)$ | Interpolation between old and new (replaces both forget and input gates) || **Candidate** | $\tilde{h}_t = \tanh(W_h [r_t \odot h_{t-1}, x_t] + b_h)$ | Proposed new state || **State update** | $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$ | Single gate controls both forgetting and input |**Key insight**: The update gate $z_t$ plays the role of **both** the forget gate and the input gate in LSTM. When $z_t \approx 0$, the old state passes through unchanged (analogous to $f_t \approx 1$ in LSTM).#### GRU vs LSTM Comparison| | GRU | LSTM ||---|---|---|| Hidden state | Single $h_t$ | Separate cell state $c_t$ and hidden state $h_t$ || Gates | 2 (reset, update) | 3 (forget, input, output) || Reset gate | Controls how much past info is used to form candidate | — || Update gate | Controls retention vs replacement | — || Forget gate | — | Controls how much old memory to keep || Input gate | — | Controls how much new info to write || Output gate | — | Controls how much memory to expose |**Parameter count** (with $H = 50$, $d = 20$):- LSTM: $4 \times H \times (H + d) = 14{,}000$- GRU: $3 \times H \times (H + d) = 10{,}500$ — **25% fewer** parameters

### 4.2 From-Scratch Implementation

In [ ]:
# ── GRU ───────────────────────────────────────────────────────────────────────class GRUModel(nn.Module):    def __init__(self, in_feature, hidden_size, n_class):        super(GRUModel, self).__init__()        self.in_feature = in_feature        self.hidden_size = hidden_size        self.n_class = n_class        self.fully_connected_1 = nn.Linear(in_feature + self.hidden_size,                                            2 * self.hidden_size)        self.fully_connected_2 = nn.Linear(in_feature + self.hidden_size,                                            self.hidden_size)        self.pred_layer = nn.Linear(self.hidden_size, self.n_class)        self.tanh = nn.Tanh()        self.sigmoid = nn.Sigmoid()    def forward(self, input, dtype=torch.float):        T = input.shape[0]        batch_size = input.shape[1]        outputs = torch.zeros(size=(T, batch_size, self.hidden_size), dtype=dtype)        state = torch.zeros(size=(batch_size, self.hidden_size), dtype=dtype)        for t in range(T):            concat = torch.cat([input[t], state], dim=1)            gates = self.fully_connected_1(concat)            r, z = gates.chunk(2, dim=1)            r_t = self.sigmoid(r)            z_t = self.sigmoid(z)            concat_reset = torch.cat([input[t], r_t * state], dim=1)            h_tilde = self.tanh(self.fully_connected_2(concat_reset))            state = (1 - z_t) * state + z_t * h_tilde            outputs[t] = state        return outputs, state    def predict(self, input_state, dtype=torch.float):        _, last_state = self.forward(input_state)        predict = self.pred_layer(last_state)        return predict

### 4.3 Training and Three-Way Comparison

In [ ]:
gru_model = GRUModel(embedding_size, hidden_size, output_size)train_and_log(gru_model, 'gru', maps)

In [ ]:
# ── Three-way comparison: RNN vs LSTM vs GRU ───────────────────────────────fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))colors = {'rnn': '#e74c3c', 'lstm': '#3498db', 'gru': '#2ecc71'}for name in ['rnn', 'lstm', 'gru']:    if name in results:        ax1.plot(results[name]['train_loss'], label=name.upper(), color=colors[name])        ax2.plot(results[name]['val_acc'], label=name.upper(), color=colors[name])ax1.set_xlabel('Step (x{})'.format(PRINT_INTERVAL))ax1.set_ylabel('Training Loss')ax1.set_title('Training Loss Comparison')ax1.legend(); ax1.grid(True, alpha=0.3)ax2.set_xlabel('Validation checkpoint')ax2.set_ylabel('Accuracy')ax2.set_title('Validation Accuracy Comparison')ax2.legend(); ax2.grid(True, alpha=0.3)plt.tight_layout(); plt.show()# Summary tableprint("\n" + "="*60)print(f"{'Model':<10} {'Params':<12} {'Final Val Acc':<15}")print("="*60)for name, model in [('RNN', rnn_model), ('LSTM', lstm_model), ('GRU', gru_model)]:    n_params = sum(p.numel() for p in model.parameters())    final_acc = results[name.lower()]['val_acc'][-1]    print(f"{name:<10} {n_params:<12,} {final_acc:<15.3f}")print("="*60)

---## 5. RNN ApplicationsRNN-based architectures have been applied to a wide range of tasks beyond simple sequence classification. This section covers three important application areas discussed in the lecture.### 5.1 Visual Question Answering (VQA)VQA systems take an **image** and a **natural language question** as input, and produce a **natural language answer**.**Example:** Given an image of a person with a banana mustache, and the question "What's the mustache made of?", the system should answer "Banana".#### VQA PipelineThe VQA pipeline involves four key components:**1. Question Encoding (Question Attention)**- Each word in the question is embedded and processed by an LSTM.- A **question attention** mechanism (using Conv → Softmax over LSTM outputs) computes a weighted combination of all word representations to form the **question vector**.- This is similar to self-attention over the question sequence.**2. Object Detection**- A pre-trained object detector (e.g., Faster R-CNN) identifies objects in the image.- Each detected object is represented by a feature vector from the CNN backbone.- This produces a set of **object vectors**.**3. Image Attention**- The question vector is used to attend over the detected objects.- Element-wise product between the question vector (tiled) and each object's CNN features.- L2 normalisation + Softmax produces attention weights over objects.- The weighted sum of object features gives the **image attention vector**.**4. Question & Image Fusion**- The question vector and image attention vector are combined via element-wise product.- Passed through FC layers → L2 normalisation → Softmax → predicted answer.This pipeline demonstrates how RNNs (for question encoding) combine with CNNs (for image features) in a multimodal architecture.

### 5.2 Reading Comprehension and Memory NetworksReading comprehension tasks require answering a question based on a set of given facts.**Example:**- A. Brian is a frog.- B. Lily is gray.- C. Brian is yellow.- D. Julius is green.- E. Greg is a frog.- **Question:** What color is Greg?#### Architecture: End-to-End Memory NetworksThe key idea: use an **attention mechanism** over a memory bank of facts.1. **Memory encoding**: Each fact is encoded into a vector using an LSTM (or embedding + positional encoding).2. **Query encoding**: The question is encoded into a query vector.3. **Attention**: The query attends over all memory vectors to select the most relevant facts.4. **Answer generation**: The attended memory representation is used to generate the answer.This is an early form of the **attention mechanism** applied to knowledge retrieval — a precursor to the Transformer's self-attention. The key insight is that **soft attention** over a memory bank allows the model to select relevant information without hard-coding retrieval rules.*Reference: End-To-End Memory Networks. S. Sukhbaatar, A. Szlam, J. Weston, R. Fergus. NIPS, 2015.*

### 5.3 Sequence-to-Sequence (Seq2seq)The Seq2seq model transforms an input sequence (source) to a new one (target), where both sequences can be of **arbitrary lengths**. Applications include machine translation, chatbots, and text summarisation.#### Encoder-Decoder ArchitectureThe model consists of two components:**Encoder**: Processes the input sequence and compresses it into a **context vector** (also called "thought vector") — the final hidden state of the encoder RNN.$$h_t^{\text{enc}} = \text{LSTM}(x_t, h_{t-1}^{\text{enc}})$$**Decoder**: Initialised with the context vector, generates the output sequence one token at a time:$$h_t^{\text{dec}} = \text{LSTM}(y_{t-1}, h_{t-1}^{\text{dec}})$$$$y_t = \text{softmax}(W_o h_t^{\text{dec}})$$**Critical limitation**: The entire source sequence must compress into a **fixed-length** context vector. For long sequences, the encoder "forgets" the beginning by the time it reaches the end. This is the **information bottleneck** that attention was designed to solve.

In [ ]:
# ── Seq2seq Encoder-Decoder (simplified for demonstration) ────────────────────class Seq2SeqEncoder(nn.Module):    def __init__(self, vocab_size, embed_dim, hidden_dim):        super().__init__()        self.embedding = nn.Embedding(vocab_size, embed_dim)        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)    def forward(self, src):        # src: (B, T_src)        embedded = self.embedding(src)          # (B, T_src, embed_dim)        outputs, (h_n, c_n) = self.lstm(embedded)        return outputs, h_n, c_n               # outputs for attention, (h_n, c_n) as contextclass Seq2SeqDecoder(nn.Module):    def __init__(self, vocab_size, embed_dim, hidden_dim):        super().__init__()        self.embedding = nn.Embedding(vocab_size, embed_dim)        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)        self.fc_out = nn.Linear(hidden_dim, vocab_size)    def forward(self, tgt, h_0, c_0):        # tgt: (B, T_tgt) — teacher-forced target tokens        embedded = self.embedding(tgt)        outputs, (h_n, c_n) = self.lstm(embedded, (h_0, c_0))        predictions = self.fc_out(outputs)     # (B, T_tgt, vocab_size)        return predictions# Demonstrate shape flowvocab_size, embed_dim, hidden_dim = 100, 32, 64encoder = Seq2SeqEncoder(vocab_size, embed_dim, hidden_dim)decoder = Seq2SeqDecoder(vocab_size, embed_dim, hidden_dim)src = torch.randint(0, vocab_size, (2, 8))    # batch=2, source length=8tgt = torch.randint(0, vocab_size, (2, 6))    # target length=6enc_outputs, h_n, c_n = encoder(src)dec_outputs = decoder(tgt, h_n, c_n)print(f"Source:          {src.shape}  (B, T_src)")print(f"Encoder outputs: {enc_outputs.shape}  (B, T_src, H)")print(f"Context vector:  h={h_n.shape}, c={c_n.shape}")print(f"Decoder output:  {dec_outputs.shape}  (B, T_tgt, vocab_size)")

### 5.4 Cross-Attention in Neural Machine TranslationThe **attention mechanism** (Bahdanau et al., 2014) solves the information bottleneck by allowing the decoder to look at **all** encoder hidden states at each decoding step, not just the final context vector.#### Mathematical FormulationGiven source sequence $\mathbf{x} = [x_1, \ldots, x_n]$ and target sequence $\mathbf{y} = [y_1, \ldots, y_m]$:The encoder (typically a **bidirectional RNN**) produces hidden states $h_1, \ldots, h_n$.At each decoder step $t$, the decoder computes:1. **Alignment scores**: How well each source position $i$ matches the current decoder state $s_{t-1}$:$$\text{score}(s_{t-1}, h_i) = v_a^\top \tanh(W_a [s_{t-1}; h_i])$$2. **Attention weights** (via softmax):$$\alpha_{t,i} = \frac{\exp(\text{score}(s_{t-1}, h_i))}{\sum_{i'=1}^{n} \exp(\text{score}(s_{t-1}, h_{i'}))}$$3. **Context vector** (weighted sum of encoder states):$$c_t = \sum_{i=1}^{n} \alpha_{t,i} h_i$$4. **Decoder state update** (using context as additional input):$$s_t = f(s_{t-1}, y_{t-1}, c_t)$$The set of attention weights $\{\alpha_{t,i}\}$ forms an **alignment matrix** that explicitly shows the correspondence between source and target positions. This is a form of **cross-attention** — the query comes from one sequence (decoder) and the keys/values come from another (encoder).**Key difference from self-attention**: In cross-attention, Q comes from the decoder and K, V come from the encoder. In self-attention (which we'll see in the Transformer), Q, K, V all come from the same sequence.

In [ ]:
# ── Cross-Attention (Bahdanau-style) ──────────────────────────────────────────class BahdanauAttention(nn.Module):    """Additive (Bahdanau) attention for Seq2seq."""    def __init__(self, hidden_dim):        super().__init__()        self.W_a = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)  # [s; h] -> hidden        self.v_a = nn.Linear(hidden_dim, 1, bias=False)               # hidden -> score    def forward(self, decoder_state, encoder_outputs):        """        Args:            decoder_state:   (B, H) — current decoder hidden state            encoder_outputs: (B, T_src, H) — all encoder hidden states        Returns:            context: (B, H) — attended encoder representation            weights: (B, T_src) — attention weights (alignment scores)        """        T_src = encoder_outputs.size(1)        # Repeat decoder state for each source position        s = decoder_state.unsqueeze(1).expand(-1, T_src, -1)  # (B, T_src, H)        # Concatenate and compute scores        combined = torch.cat([s, encoder_outputs], dim=2)      # (B, T_src, 2H)        energy = torch.tanh(self.W_a(combined))                # (B, T_src, H)        scores = self.v_a(energy).squeeze(2)                   # (B, T_src)        weights = F.softmax(scores, dim=1)                     # (B, T_src)        # Weighted sum of encoder outputs        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)  # (B, H)        return context, weights# Demonstrateattn = BahdanauAttention(hidden_dim=64)dec_state = torch.randn(2, 64)               # batch=2, H=64enc_out = torch.randn(2, 8, 64)              # T_src=8context, weights = attn(dec_state, enc_out)print(f"Decoder state:   {dec_state.shape}")print(f"Encoder outputs: {enc_out.shape}")print(f"Context vector:  {context.shape}")print(f"Attention weights: {weights.shape}  (sum={weights[0].sum().item():.4f})")# Visualise alignment for one exampleplt.figure(figsize=(6, 2))plt.bar(range(8), weights[0].detach().numpy())plt.xlabel('Source position')plt.ylabel('Attention weight')plt.title('Cross-Attention Weights (one decoder step)')plt.tight_layout(); plt.show()

In [ ]:
# ── Seq2seq with Attention (complete model) ──────────────────────────────────class Seq2SeqWithAttention(nn.Module):    def __init__(self, src_vocab, tgt_vocab, embed_dim, hidden_dim):        super().__init__()        self.encoder = Seq2SeqEncoder(src_vocab, embed_dim, hidden_dim)        self.attention = BahdanauAttention(hidden_dim)        self.tgt_embedding = nn.Embedding(tgt_vocab, embed_dim)        # Decoder LSTM takes embedding + context as input        self.decoder_lstm = nn.LSTMCell(embed_dim + hidden_dim, hidden_dim)        self.fc_out = nn.Linear(hidden_dim, tgt_vocab)    def forward(self, src, tgt):        B, T_tgt = tgt.shape        enc_outputs, h_n, c_n = self.encoder(src)        h, c = h_n.squeeze(0), c_n.squeeze(0)        outputs = []        for t in range(T_tgt):            tgt_emb = self.tgt_embedding(tgt[:, t])       # (B, embed_dim)            context, attn_w = self.attention(h, enc_outputs)            lstm_input = torch.cat([tgt_emb, context], dim=1)            h, c = self.decoder_lstm(lstm_input, (h, c))            out = self.fc_out(h)                           # (B, tgt_vocab)            outputs.append(out)        return torch.stack(outputs, dim=1)                 # (B, T_tgt, tgt_vocab)# Demonstratemodel_attn = Seq2SeqWithAttention(100, 80, embed_dim=32, hidden_dim=64)src = torch.randint(0, 100, (2, 10))tgt = torch.randint(0, 80, (2, 7))out = model_attn(src, tgt)print(f"Seq2seq+Attention output: {out.shape}  (B, T_tgt, tgt_vocab)")

---## 6. Transformer### 6.0 The Limits of RecurrenceEven with gating and attention, recurrent models share a fundamental constraint: **sequential processing**. Token $t$ must wait for token $t-1$ to finish. This creates three problems:1. **No parallelism**: Training time scales linearly with sequence length. GPUs can't process all time steps simultaneously.2. **Information bottleneck**: Even with attention, the decoder state is still a sequential bottleneck.3. **Long-range dependencies are still hard**: Even LSTM struggles when $T > 200$-$500$.What if we could let **every token attend to every other token** simultaneously, without any recurrence?> This is the core idea behind the **Transformer** (Vaswani et al., "Attention is All You Need", 2017).The Transformer was a revolutionary breakthrough — GPT-3, for example, is the 3rd version of Generative Pre-Trained Transformer released by OpenAI, capable of tasks like generating web application code from a natural language description.

### 6.1 Self-Attention MechanismSelf-attention (also called intra-attention) is an attention mechanism relating different positions of a **single sequence** to compute a representation of that same sequence. Unlike cross-attention (§5.4), where Q comes from one sequence and K, V from another, in self-attention **Q, K, V all come from the same sequence**.**Intuition**: Consider the sentence *"The FBI agent who was responsible is now retired."* The verb *"is"* depends on the subject *"agent"* (not the closer *"responsible"*). Self-attention can capture this directly.#### Query, Key, and ValueThe key/value/query concepts come from **retrieval systems**. When you type a query to search for a video on YouTube, the search engine maps your query against a set of keys (video title, description) associated with candidate videos in the database, then presents the best matched videos (values).For each input token $x_i$, we compute three vectors via learned linear projections:$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$where $X \in \mathbb{R}^{T \times d_{\text{model}}}$ and $W^Q, W^K, W^V \in \mathbb{R}^{d_{\text{model}} \times d_k}$.- **Query** ($Q$): "What am I looking for?" (to match others)- **Key** ($K$): "What do I contain?" (to be matched)- **Value** ($V$): "What information do I provide?" (information to be extracted)Since $Q_i = W^Q a_i$ and $K_j = W^K a_j$, the dot product $Q_i \cdot K_j$ measures the **similarity** between the projections of tokens $i$ and $j$. Self-attention is exactly about relating different tokens to each other!

#### Scaled Dot-Product AttentionThe attention mechanism computes:$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$Step by step:1. **Similarity scores**: $S = QK^\top \in \mathbb{R}^{T \times T}$ — dot product between each query and all keys.2. **Scaling**: Divide by $\sqrt{d_k}$ to prevent large magnitudes that would push softmax into saturation.3. **Softmax**: Normalise each row to get attention weights $A \in \mathbb{R}^{T \times T}$ where each row sums to 1.4. **Weighted sum**: $\text{Output} = AV \in \mathbb{R}^{T \times d_k}$ — each output token is a mixture of all value vectors, weighted by attention.This can also be understood as **soft addressing**: the query $Q$ computes similarity with all keys $K$ to determine how much of each value $V$ to retrieve.#### Why $\frac{1}{\sqrt{d_k}}$?Without scaling, each entry of $QK^\top$ is a sum of $d_k$ products. If the entries of $Q$ and $K$ have unit variance, the variance of each dot product is $d_k$, and the standard deviation grows like $\sqrt{d_k}$. Large magnitudes push softmax into near-one-hot outputs where gradients vanish. Dividing by $\sqrt{d_k}$ keeps the variance at approximately 1.#### Key advantage over RNNsThe attention matrix $A$ connects **every pair of tokens** directly — information doesn't need to travel through a chain of hidden states. The path length between any two tokens is $O(1)$, compared to $O(T)$ in RNNs.All output tokens $b_1, b_2, \ldots, b_T$ can be computed **in parallel** — no sequential dependency!

#### Matrix FormThe full computation in matrix notation:$$Q = W^Q I, \quad K = W^K I, \quad V = W^V I$$$$A' = K^\top Q \quad \text{(attention scores)}$$$$\hat{A}' = \text{softmax}(A' / \sqrt{d_k}) \quad \text{(normalised weights)}$$$$O = V \hat{A}' \quad \text{(output)}$$Or more compactly:$$O = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

In [ ]:
# ── Scaled Dot-Product Attention ──────────────────────────────────────────────def scaled_dot_product_attention(Q, K, V, mask=None):    """    Args:        Q, K, V: (B, H, T, d_k) tensors        mask:    (T, T) or broadcastable tensor, or None. 1 = keep, 0 = block.    Returns:        output:       (B, H, T, d_k)        attn_weights: (B, H, T, T), each row sums to 1    """    d_k = Q.size(-1)    scores = torch.matmul(Q, K.transpose(-2, -1))          # (B, H, T, T)    scores = scores / math.sqrt(d_k)                        # scale    if mask is not None:        scores = scores.masked_fill(mask == 0, float("-inf"))    attn_weights = F.softmax(scores, dim=-1)                # (B, H, T, T)    output = torch.matmul(attn_weights, V)                  # (B, H, T, d_k)    return output, attn_weights

In [ ]:
# ── Verification ──────────────────────────────────────────────────────────────torch.manual_seed(42)B, H, T, d_k = 2, 4, 6, 16Q = torch.randn(B, H, T, d_k)K = torch.randn(B, H, T, d_k)V = torch.randn(B, H, T, d_k)out, w = scaled_dot_product_attention(Q, K, V)assert out.shape == (B, H, T, d_k), f"output shape wrong: {out.shape}"assert w.shape == (B, H, T, T), f"attn shape wrong: {w.shape}"assert torch.allclose(w.sum(dim=-1), torch.ones(B, H, T), atol=1e-5), "rows must sum to 1"print(f"Output shape: {out.shape}")print(f"Attention shape: {w.shape}")print("Row sums: all 1.0 ✓")# Visualise with and without causal maskcausal = torch.tril(torch.ones(T, T))out_m, w_m = scaled_dot_product_attention(Q, K, V, mask=causal)fig, axes = plt.subplots(1, 2, figsize=(9, 4))for ax, weights, title in [    (axes[0], w.detach()[0, 0], "No mask"),    (axes[1], w_m.detach()[0, 0], "Causal mask"),]:    im = ax.imshow(weights, cmap="viridis", vmin=0, vmax=weights.max().item())    ax.set_title(title); ax.set_xlabel("key position"); ax.set_ylabel("query position")    fig.colorbar(im, ax=ax, fraction=0.046)plt.tight_layout(); plt.show()

### 6.2 Multi-Head Self-AttentionA single attention head can only focus on **one type of relationship** at a time. Multi-head attention runs $h$ parallel attention computations with **different learned projections**, then concatenates the results. Each head can specialise: one might attend to syntactic structure, another to semantic similarity.$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O$$$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$**Design choice**: $d_k = d_v = d_{\text{model}} / h$. Since each head operates on a smaller dimension, the total computation is roughly the same as single-head attention with full $d_{\text{model}}$ — but the model gains **multiple perspectives**.According to the original paper: *"multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this."*

In [ ]:
# ── Multi-Head Attention ─────────────────────────────────────────────────────class MultiHeadAttention(nn.Module):    def __init__(self, d_model, num_heads):        super().__init__()        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"        self.d_k = d_model // num_heads        self.num_heads = num_heads        self.W_q = nn.Linear(d_model, d_model)        self.W_k = nn.Linear(d_model, d_model)        self.W_v = nn.Linear(d_model, d_model)        self.W_o = nn.Linear(d_model, d_model)    def forward(self, Q, K, V, mask=None):        B, T, _ = Q.size()        Q = self.W_q(Q).view(B, T, self.num_heads, self.d_k).transpose(1, 2)        K = self.W_k(K).view(B, T, self.num_heads, self.d_k).transpose(1, 2)        V = self.W_v(V).view(B, T, self.num_heads, self.d_k).transpose(1, 2)        out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)        out = out.transpose(1, 2).contiguous().view(B, T, -1)        return self.W_o(out), attn_weights

In [ ]:
# ── Verification ──────────────────────────────────────────────────────────────torch.manual_seed(42)d_model, num_heads, T = 64, 4, 6x = torch.randn(2, T, d_model)mha = MultiHeadAttention(d_model, num_heads)out, attn = mha(x, x, x)  # self-attention: Q=K=V=xprint(f"Input shape:     {x.shape}")print(f"Output shape:    {out.shape}")print(f"Attention shape: {attn.shape}  (batch, heads, T, T)")

### 6.3 Positional EncodingUnlike RNNs, the Transformer processes all tokens **simultaneously** — it has no notion of sequence order. Without positional information, the model treats "dog bites man" identically to "man bites dog"!We inject position information by adding a **positional encoding** to the input embeddings:$$\text{input}_i = \text{embedding}(x_i) + \text{PE}(i)$$#### Sinusoidal Positional EncodingThe original Transformer uses fixed sine/cosine functions:$$PE_{pos, 2i} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$$$PE_{pos, 2i+1} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$where $pos$ is the token position in the sequence, $i$ is the dimension index, and $d$ is the embedding dimension.**Why sine and cosine?** For any fixed offset $k$, $PE_{pos+k}$ can be expressed as a **linear function** of $PE_{pos}$. This gives the model a systematic way to compute **relative positions** from absolute encodings.**Frequency interpretation**: Lower dimensions vary faster (high frequency), while higher dimensions vary more slowly (low frequency). This allows the model to encode both **local** (nearby tokens) and **global** (distant tokens) position patterns.Equivalently, each position $pos$ is represented by appending a one-hot vector $p_i$ that indicates position $i$, and the positional encoding is a learned (or fixed) transformation of that one-hot vector:$$\text{PE}(pos) = W^{\text{pos}} \cdot \text{one-hot}(pos)$$

In [ ]:
# ── Positional Encoding ──────────────────────────────────────────────────────class PositionalEncoding(nn.Module):    def __init__(self, d_model, dropout=0.1, max_len=5000):        super(PositionalEncoding, self).__init__()        self.dropout = nn.Dropout(p=dropout)        pe = torch.zeros(max_len, d_model)        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)        div_term = torch.exp(torch.arange(0, d_model, 2).float()                             * (-math.log(10000.0) / d_model))        pe[:, 0::2] = torch.sin(position * div_term)        pe[:, 1::2] = torch.cos(position * div_term)        pe = pe.unsqueeze(0).transpose(0, 1)        self.register_buffer('pe', pe)    def forward(self, x):        x = x + self.pe[:x.size(0), :]        return self.dropout(x)

In [ ]:
# ── Visualise positional encoding ────────────────────────────────────────────pe_module = PositionalEncoding(d_model=128, dropout=0.0)pe_values = pe_module.pe[:100, 0, :].numpy()fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Heatmapim = axes[0].imshow(pe_values, aspect='auto', cmap='RdBu', interpolation='nearest')fig.colorbar(im, ax=axes[0], label='Value')axes[0].set_xlabel('Embedding Dimension')axes[0].set_ylabel('Position')axes[0].set_title('Positional Encoding Heatmap')# Individual dimension curvesfor dim in [0, 1, 4, 5, 20, 21]:    axes[1].plot(pe_values[:, dim], label=f'dim {dim}', alpha=0.8)axes[1].set_xlabel('Position')axes[1].set_ylabel('Value')axes[1].set_title('PE values for selected dimensions')axes[1].legend(fontsize=8)axes[1].grid(True, alpha=0.3)plt.tight_layout(); plt.show()print("Notice: lower dimensions oscillate faster (local patterns), higher dimensions slower (global patterns)")

### 6.4 Transformer Encoder BlockA single Transformer encoder block consists of:1. **Multi-Head Self-Attention** → 2. **Add & LayerNorm** → 3. **Feed-Forward Network (FFN)** → 4. **Add & LayerNorm**Key design choices:- **Residual connections** (Add): Same principle as ResNet — allow gradients to flow directly through the network, enabling deeper stacking.- **Layer Normalisation** (not Batch Normalisation): Normalises across **features** for each sample independently, making it more stable for **variable-length** sequences where batch statistics would be unreliable.- **Feed-Forward Network**: Expands then contracts — $d_{\text{model}} \to d_{ff} \to d_{\text{model}}$, typically with $d_{ff} = 4 \times d_{\text{model}}$. Applied to each position **independently** (position-wise), providing additional non-linear transformation capacity.

In [ ]:
# ── Transformer Encoder Block ────────────────────────────────────────────────class TransformerEncoderBlock(nn.Module):    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):        super().__init__()        self.attention = MultiHeadAttention(d_model, num_heads)        self.norm1 = nn.LayerNorm(d_model)        self.ffn = nn.Sequential(            nn.Linear(d_model, d_ff),            nn.ReLU(),            nn.Linear(d_ff, d_model)        )        self.norm2 = nn.LayerNorm(d_model)        self.dropout = nn.Dropout(dropout)    def forward(self, x, mask=None):        attn_out, attn_weights = self.attention(x, x, x, mask)        x = self.norm1(x + self.dropout(attn_out))        ffn_out = self.ffn(x)        x = self.norm2(x + self.dropout(ffn_out))        return x, attn_weights

In [ ]:
# ── Verification ──────────────────────────────────────────────────────────────torch.manual_seed(42)block = TransformerEncoderBlock(d_model=64, num_heads=4, d_ff=256)x = torch.randn(2, 10, 64)out, attn = block(x)print(f"Input:     {x.shape}")print(f"Output:    {out.shape}")print(f"Attention: {attn.shape}")

### 6.5 MaskingFor autoregressive tasks (e.g., language modelling), we must prevent the model from "seeing the future". We apply a **causal mask** — a lower-triangular matrix where future positions are set to $-\infty$ before softmax.$$\text{mask}_{ij} = \begin{cases} 0 & \text{if } j \leq i \quad (\text{allowed}) \\\\ -\infty & \text{if } j > i \quad (\text{blocked}) \end{cases}$$After softmax, the $-\infty$ entries become 0, so each token can only attend to itself and earlier tokens.**Why $-\infty$ before softmax, not 0 after?** Setting scores to $-\infty$ before softmax ensures the blocked positions get exactly zero weight and the remaining weights still sum to 1 (proper probability distribution). Zeroing out weights after softmax would break the normalisation.

In [ ]:
# ── Causal mask generation ───────────────────────────────────────────────────def generate_square_subsequent_mask(sz):    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))    return maskmask = generate_square_subsequent_mask(6)print("Causal mask (0 = allowed, -inf = blocked):")print(mask)

### 6.6 Full Transformer Architecture: Encoder-DecoderThe complete Transformer consists of an **encoder** and a **decoder**, designed for sequence-to-sequence tasks like machine translation.#### EncoderA stack of $N$ identical encoder blocks (typically $N = 6$). Each block contains:1. Multi-head **self-attention** (attend to the input sequence)2. Position-wise FFNThe encoder processes the entire source sequence in parallel and produces a set of representations.#### DecoderA stack of $N$ identical decoder blocks, each containing:1. **Masked** multi-head self-attention (attend to previously generated tokens only)2. Multi-head **cross-attention** (attend to encoder output — Q from decoder, K and V from encoder)3. Position-wise FFNKey differences between encoder and decoder attention:- **Encoder self-attention**: Each token attends to all positions (bidirectional)- **Decoder masked self-attention**: Each token attends only to previous positions (causal/autoregressive)- **Decoder cross-attention**: Decoder queries attend to all encoder outputs#### Layer NormalisationThe Transformer uses **Layer Norm** instead of Batch Norm. In the original paper, it is applied **after** the residual addition (Post-LN). Many modern implementations apply it **before** (Pre-LN), which tends to stabilise training for very deep models.#### Complete Forward Pass (Translation Example)Using Chinese-to-English translation as example:1. **Input**: "神经网络" (source tokens)2. **Encoder**: Process all source tokens → encoder representations3. **Decoder input**: `<BOS>` (beginning of sentence token)4. **Decoder**: Attend to encoder output + previously generated tokens → predict "Neural"5. **Repeat**: Feed "Neural" back → predict "Network" → ... until `<EOS>`

### 6.7 Attention VisualisationAttention weights provide interpretable insights into what the model is "looking at". Each head in multi-head attention can learn different patterns:- Some heads attend to **adjacent tokens** (local context)- Some heads attend to **syntactically related** tokens (e.g., subject-verb agreement)- Some heads attend to **semantically related** tokens (e.g., coreference resolution)For example, in a Transformer trained on English-to-French translation, the encoder self-attention for the word "it" shows that certain heads learn to resolve coreference — attending back to the noun that "it" refers to, even across long distances.The alignment matrix between source and target in cross-attention closely mirrors human-interpretable word alignments in translation tasks.

In [ ]:
# ── Attention Visualisation ───────────────────────────────────────────────────def visualise_attention(sentence_tokens, attn_weights, layer_name=""):    """Visualise attention weights for all heads.    Args:        sentence_tokens: list of token strings        attn_weights: (num_heads, T, T) tensor    """    num_heads = attn_weights.shape[0]    fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))    if num_heads == 1:        axes = [axes]    for h in range(num_heads):        ax = axes[h]        im = ax.imshow(attn_weights[h].detach().numpy(), cmap='Blues', vmin=0, vmax=1)        ax.set_xticks(range(len(sentence_tokens)))        ax.set_yticks(range(len(sentence_tokens)))        ax.set_xticklabels(sentence_tokens, rotation=45, ha='right', fontsize=8)        ax.set_yticklabels(sentence_tokens, fontsize=8)        ax.set_title(f'Head {h+1}', fontsize=10)        ax.set_xlabel('Key')        ax.set_ylabel('Query')    plt.suptitle(f'Multi-Head Attention Weights {layer_name}', fontsize=12)    plt.tight_layout()    plt.show()# Create a simple example and visualisetorch.manual_seed(123)tokens = ["The", "cat", "sat", "on", "the", "mat"]T = len(tokens)d_model, num_heads = 32, 4mha_viz = MultiHeadAttention(d_model, num_heads)x = torch.randn(1, T, d_model)_, attn = mha_viz(x, x, x)visualise_attention(tokens, attn[0], "(random init — patterns emerge after training)")

### 6.8 Language Modelling with nn.TransformerEncoderNow we put everything together: train a Transformer on a **language modelling** task using the Penn Treebank dataset. The model predicts the next word given all previous words.We use PyTorch's built-in `nn.TransformerEncoder` (which internally uses the same components we built from scratch above).

In [ ]:
# ── Transformer Language Model ───────────────────────────────────────────────from torch.nn import TransformerEncoder, TransformerEncoderLayerclass TransformerLM(nn.Module):    def __init__(self, ntoken, ninp, nhead, nhid, nlayers, dropout=0.5):        super(TransformerLM, self).__init__()        self.model_type = 'Transformer'        self.pos_encoder = PositionalEncoding(ninp, dropout)        encoder_layers = TransformerEncoderLayer(ninp, nhead, nhid, dropout)        self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)        self.encoder = nn.Embedding(ntoken, ninp)        self.ninp = ninp        self.decoder = nn.Linear(ninp, ntoken)        self.init_weights()    def generate_square_subsequent_mask(self, sz):        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))        return mask    def init_weights(self):        initrange = 0.1        self.encoder.weight.data.uniform_(-initrange, initrange)        self.decoder.bias.data.zero_()        self.decoder.weight.data.uniform_(-initrange, initrange)    def forward(self, src, src_mask):        src = self.encoder(src) * math.sqrt(self.ninp)        src = self.pos_encoder(src)        output = self.transformer_encoder(src, src_mask)        output = self.decoder(output)        return output

In [ ]:
# ── Data loading: Penn Treebank ──────────────────────────────────────────────# Requires: pip install torchtext torchdata portalocker>=2.0.0from torchtext.datasets import PennTreebankfrom torchtext.data.utils import get_tokenizerfrom torchtext.vocab import build_vocab_from_iteratortrain_iter = PennTreebank(split='train')tokenizer = get_tokenizer('basic_english')vocab = build_vocab_from_iterator(map(tokenizer, train_iter), specials=["<unk>"])vocab.set_default_index(vocab["<unk>"])def data_process(raw_text_iter):    data = [torch.tensor(vocab(tokenizer(item)), dtype=torch.long) for item in raw_text_iter]    return torch.cat(tuple(filter(lambda t: t.numel() > 0, data)))train_iter, val_iter, test_iter = PennTreebank()train_data_lm = data_process(train_iter)val_data_lm = data_process(val_iter)test_data_lm = data_process(test_iter)device = torch.device("cuda" if torch.cuda.is_available() else "cpu")def batchify(data, bsz):    nbatch = data.size(0) // bsz    data = data.narrow(0, 0, nbatch * bsz)    data = data.view(bsz, -1).t().contiguous()    return data.to(device)lm_batch_size = 20eval_batch_size = 10train_data_lm = batchify(train_data_lm, lm_batch_size)val_data_lm = batchify(val_data_lm, eval_batch_size)test_data_lm = batchify(test_data_lm, eval_batch_size)print(f"Vocabulary size: {len(vocab)}")print(f"Train data shape: {train_data_lm.shape}")

In [ ]:
# ── Batching for language modelling ──────────────────────────────────────────bptt = 35def get_batch(source, i):    seq_len = min(bptt, len(source) - 1 - i)    data = source[i:i+seq_len]    target = source[i+1:i+1+seq_len].reshape(-1)    return data, target

In [ ]:
# ── Model instantiation ──────────────────────────────────────────────────────ntokens = len(vocab)emsize = 200nhid = 200nlayers = 2nhead = 2dropout = 0.2transformer_model = TransformerLM(ntokens, emsize, nhead, nhid, nlayers, dropout).to(device)n_params = sum(p.numel() for p in transformer_model.parameters())print(f"Transformer parameters: {n_params:,}")

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────criterion = nn.CrossEntropyLoss()lr = 5.0optimizer_lm = torch.optim.SGD(transformer_model.parameters(), lr=lr)scheduler = torch.optim.lr_scheduler.StepLR(optimizer_lm, 1.0, gamma=0.95)def train_lm(epoch):    transformer_model.train()    total_loss = 0.    start_time = time.time()    src_mask = transformer_model.generate_square_subsequent_mask(bptt).to(device)    for batch_idx, i in enumerate(range(0, train_data_lm.size(0) - 1, bptt)):        data, targets = get_batch(train_data_lm, i)        optimizer_lm.zero_grad()        if data.size(0) != bptt:            src_mask = transformer_model.generate_square_subsequent_mask(data.size(0)).to(device)        output = transformer_model(data, src_mask)        loss = criterion(output.view(-1, ntokens), targets)        loss.backward()        torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), 0.5)        optimizer_lm.step()        total_loss += loss.item()        log_interval = 200        if batch_idx % log_interval == 0 and batch_idx > 0:            cur_loss = total_loss / log_interval            elapsed = time.time() - start_time            print(f'| epoch {epoch:3d} | {batch_idx:5d}/{len(train_data_lm) // bptt:5d} batches | '                  f'lr {scheduler.get_last_lr()[0]:.2f} | ms/batch {elapsed * 1000 / log_interval:.2f} | '                  f'loss {cur_loss:.2f} | ppl {math.exp(cur_loss):.2f}')            total_loss = 0            start_time = time.time()def evaluate_lm(eval_model, data_source):    eval_model.eval()    total_loss = 0.    src_mask = transformer_model.generate_square_subsequent_mask(bptt).to(device)    with torch.no_grad():        for i in range(0, data_source.size(0) - 1, bptt):            data, targets = get_batch(data_source, i)            if data.size(0) != bptt:                src_mask = transformer_model.generate_square_subsequent_mask(data.size(0)).to(device)            output = eval_model(data, src_mask)            output_flat = output.view(-1, ntokens)            total_loss += len(data) * criterion(output_flat, targets).item()    return total_loss / (len(data_source) - 1)

In [ ]:
# ── Train for 3 epochs ───────────────────────────────────────────────────────best_val_loss = float("inf")epochs = 3best_model = Nonefor epoch in range(1, epochs + 1):    epoch_start_time = time.time()    train_lm(epoch)    val_loss = evaluate_lm(transformer_model, val_data_lm)    print('-' * 89)    print(f'| end of epoch {epoch:3d} | time: {time.time() - epoch_start_time:.2f}s | '          f'valid loss {val_loss:.2f} | valid ppl {math.exp(val_loss):.2f}')    print('-' * 89)    if val_loss < best_val_loss:        best_val_loss = val_loss        best_model = transformer_model    scheduler.step()

In [ ]:
# ── Test evaluation ──────────────────────────────────────────────────────────test_loss = evaluate_lm(best_model, test_data_lm)print('=' * 89)print(f'| End of training | test loss {test_loss:.2f} | test ppl {math.exp(test_loss):.2f}')print('=' * 89)

---## 7. Comprehensive Comparison### 7.1 Architectural Comparison| Feature | RNN | LSTM | GRU | Transformer ||---|---|---|---|---|| **Memory mechanism** | Hidden state | Cell state + hidden state | Gated hidden state | Attention weights || **Gates** | 0 | 3 (forget, input, output) | 2 (reset, update) | 0 (learned projections) || **Sequential processing** | Yes | Yes | Yes | **No** || **Long-range dependencies** | Poor | Good | Good | **Excellent** || **Vanishing gradient** | Severe | Mitigated (additive cell) | Mitigated (additive state) | **None** (direct attention) || **Bidirectional** | Requires 2x cost | Requires 2x cost | Requires 2x cost | **Native** || **Computational complexity** | $O(T \cdot d^2)$ | $O(T \cdot d^2)$ | $O(T \cdot d^2)$ | $O(T^2 \cdot d)$ || **Maximum path length** | $O(T)$ | $O(T)$ | $O(T)$ | $O(1)$ || **Parallelisable (training)** | No | No | No | Yes |### 7.2 When to Use What- **RNN**: Rarely used in practice. Important for understanding the fundamentals.- **LSTM**: When you need explicit forget/remember control, or when the task requires modelling very long sequences with precise gating.- **GRU**: Good default for recurrent tasks. Fewer parameters than LSTM, often similar performance.- **Transformer**: Default choice for most sequence tasks when computational resources allow. Scales well with parallelism, excels at long-range dependencies. Note: $O(T^2)$ attention cost can be prohibitive for very long sequences ($T > 10{,}000$).### 7.3 The Evolution of Attention| Model | Attention Type | Query Source | Key/Value Source ||---|---|---|---|| Memory Networks | Soft addressing | Question | Fact memory || Seq2seq + Attention | Cross-attention | Decoder state | Encoder outputs || Transformer (encoder) | Self-attention | Same sequence | Same sequence || Transformer (decoder) | Self + Cross | Same seq / Decoder | Same seq / Encoder |

---## 8. Exam-style QuestionsThree medium-to-hard short-answer questions. Each question connects Transformer content back to earlier weeks. Try each on paper first — answer sketches follow in collapsed cells.

### Q1 — Why $\sqrt{d_k}$? (Week 3 connection: softmax / gradient flow)Scaled dot-product attention computes$$\text{Attn}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V.$$Suppose the entries of $Q$ and $K$ are i.i.d. with zero mean and unit variance.**(a)** Show that each entry of $Q K^\top$ has variance $d_k$.**(b)** Explain what happens to the output of the softmax as $d_k \to \infty$ **without** the $\sqrt{d_k}$ scaling.**(c)** Using what you know about the softmax Jacobian, explain why this is a problem for gradient-based training, and argue why dividing by $\sqrt{d_k}$ — as opposed to $d_k$ or $1$ — is the right fix.

<details><summary><strong>Answer sketch — Q1</strong> (open after attempting)</summary>**(a)** Let $q, k \in \mathbb{R}^{d_k}$ be a query/key pair. Their inner product is $\langle q, k \rangle = \sum_{i=1}^{d_k} q_i k_i$. Each $q_i k_i$ has mean $\mathbb{E}[q_i]\mathbb{E}[k_i] = 0$ and variance $\mathrm{Var}(q_i)\mathrm{Var}(k_i) = 1$, and the terms are independent. Therefore $\mathrm{Var}(\langle q, k \rangle) = d_k$ and the standard deviation grows like $\sqrt{d_k}$.**(b)** Without scaling, the logits fed into softmax have magnitude on the order of $\sqrt{d_k}$. For large $d_k$, the largest logit dominates and the softmax output concentrates on a single position — it approaches a one-hot distribution.**(c)** The Jacobian of softmax is $J_{ij} = p_i(\delta_{ij} - p_j)$. When the output is near one-hot ($p \approx e_{i^\star}$), every entry of $J$ is near zero, so **gradients vanish into $Q$ and $K$**. The fix needs to keep the *variance* of the logits at $O(1)$, which is exactly what dividing by $\sqrt{d_k}$ achieves. Dividing by $d_k$ instead would shrink the standard deviation to $1/\sqrt{d_k} \to 0$, over-flattening the softmax and destroying the model's ability to *select* anything — the opposite failure mode.</details>

### Q2 — RNN vs. self-attention: FLOPs, path length, parallelism (Week 5 connection: complexity analysis)Consider processing a sequence of length $T$ with hidden/model dimension $d$.**(a)** Give the time complexity of one forward pass through (i) a single vanilla RNN layer and (ii) a single self-attention layer.**(b)** Give the **maximum path length** between any two tokens in each model.**(c)** Which of the two architectures can be parallelised across the time dimension during training, and why does the other fundamentally cannot?**(d)** For what regime of $T$ and $d$ is self-attention cheaper than an RNN in raw FLOPs?

<details><summary><strong>Answer sketch — Q2</strong> (open after attempting)</summary>**(a)** RNN per step: one matmul of size $O(d^2)$, repeated for $T$ steps → $O(T d^2)$. Self-attention: the $Q K^\top$ product and the attention-weighted sum of $V$ cost $O(T^2 d)$; the $Q/K/V/O$ projections cost $O(T d^2)$. Total: $O(T^2 d + T d^2)$.**(b)** RNN: $O(T)$ — information from token 1 reaches token $T$ only by traversing $T-1$ recurrent steps. Self-attention: $O(1)$ — every token attends directly to every other token in a single layer.**(c)** Self-attention can be fully parallelised across the time axis because the computation at position $t$ does **not** depend on the representation at position $t-1$; all $T$ positions are computed from the same input tensor. The RNN hidden state satisfies $h_t = f(h_{t-1}, x_t)$, so computing $h_t$ requires $h_{t-1}$ — the recurrence is inherently sequential.**(d)** Self-attention is cheaper when $T^2 d < T d^2$, i.e. when $T < d$. In practice Transformers win on *path length and parallelism*, not on asymptotic FLOPs.</details>

### Q3 — Permutation equivariance and positional encoding (Week 5/6 connection: symmetry and inductive biases)Let $\text{Attn}(X)$ denote self-attention applied to a sequence $X \in \mathbb{R}^{T \times d}$, with $Q = X W_Q$, $K = X W_K$, $V = X W_V$. Let $P \in \mathbb{R}^{T \times T}$ be any permutation matrix.**(a)** Prove that $\text{Attn}(P X) = P \cdot \text{Attn}(X)$. (I.e., self-attention is **permutation-equivariant**.)**(b)** A classmate argues:> *"Since self-attention is permutation-equivariant, a Transformer cannot distinguish 'dog bites man' from 'man bites dog'. Adding positional encodings fixes this because the positional vectors break the symmetry."*Is the conclusion correct? Is the reasoning rigorous? If not, what is the subtle gap?**(c)** For sequence modelling, is permutation equivariance a desirable inductive bias or an obstacle?

<details><summary><strong>Answer sketch — Q3</strong> (open after attempting)</summary>**(a)** With $X' = P X$ we get $Q' = P X W_Q = P Q$ and likewise $K' = P K$, $V' = P V$. Then$$Q' K'^{\top} = P Q K^\top P^\top.$$Softmax is applied **row-wise**, and row permutation commutes with any row-wise function, so$$\mathrm{softmax}\!\left(\tfrac{Q' K'^\top}{\sqrt{d_k}}\right) = P\, \mathrm{softmax}\!\left(\tfrac{Q K^\top}{\sqrt{d_k}}\right) P^\top.$$Multiplying by $V' = P V$ and using $P^\top P = I$:$$\text{Attn}(P X) = P\, \mathrm{softmax}(\cdots)\, P^\top P V = P\, \mathrm{softmax}(\cdots) V = P\, \text{Attn}(X). \quad\blacksquare$$**(b) Conclusion correct, reasoning loose.** The conclusion is right. But "adding positional encodings breaks the symmetry" is not automatic. The symmetry is only broken because PE depends on position index, not on token identity, and is **not** permuted along with the tokens. If you permuted PE together with X, you would get $\text{Attn}(P(X + \text{PE})) = P\,\text{Attn}(X + \text{PE})$ by part (a), and the problem would reappear. The gap: they did not explain *why* PE is not permuted.**(c)** It is an **obstacle**. In sequence data the *order* is meaningful — "dog bites man" ≠ "man bites dog". This is the opposite of GNNs, where permutation equivariance is desirable because node labels are arbitrary. Positional encoding exists precisely to cancel the symmetry.</details>